# LDA Topic Feature Evaluation (IEMOCAP)

Notebook-style evaluation for the LDA text families:
- `lda_topics`
- `lda_topics_context`

The workflow matches the repo's other TreeBased notebooks: load precomputed features, evaluate a small classical-model set, compare feature-family combinations, and save result artifacts.

## 1) Setup and Configuration

This section resolves the repo root, imports the shared family loader, defines evaluation combos, and sets output paths.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


cwd = Path.cwd().resolve()
REPO_ROOT = find_repo_root(cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_extraction.pipelines.pipeline_common import load_family_frame

BASE_METADATA_COLS = [
    'path',
    'session',
    'method',
    'gender',
    'emotion',
    'n_annotators',
    'agreement',
    'utt_id',
    'text',
    'split',
]

DEFAULT_COMBOS = [
    ('lda_topics',),
    ('lda_topics_context',),
    ('lda_topics', 'mfcc_normalized'),
    ('lda_topics', 'prosody_energy', 'prosody_pitch'),
    ('lda_topics', 'mfcc_normalized', 'prosody_energy', 'prosody_pitch'),
    ('lda_topics_context', 'mfcc_normalized', 'prosody_energy', 'prosody_pitch'),
]

EXCLUDED_EMOTIONS = {'sur', 'fea', 'oth', 'dis'}
RANDOM_STATE = 42
MIN_TRAIN_ROWS = 30
MIN_TEST_ROWS = 10
OUT_DIR = REPO_ROOT / 'feature_test' / 'TreeBased' / 'artifacts' / 'text_lda_topics_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {REPO_ROOT}')
print(f'Output dir: {OUT_DIR}')
print(f'Default combos: {DEFAULT_COMBOS}')


Repo root: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition
Output dir: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/feature_test/TreeBased/artifacts/text_lda_topics_eval
Default combos: [('lda_topics',), ('lda_topics_context',), ('lda_topics', 'mfcc_normalized'), ('lda_topics', 'prosody_energy', 'prosody_pitch'), ('lda_topics', 'mfcc_normalized', 'prosody_energy', 'prosody_pitch'), ('lda_topics_context', 'mfcc_normalized', 'prosody_energy', 'prosody_pitch')]


## 2) Helper Functions

Define the model zoo, metrics, family-frame merge logic, and combo evaluator.

In [2]:
def build_model_zoo(random_state: int) -> dict[str, Any]:
    return {
        'logreg': Pipeline(
            steps=[
                ('scaler', StandardScaler()),
                (
                    'clf',
                    LogisticRegression(
                        max_iter=2500,
                        class_weight='balanced',
                        random_state=random_state,
                    ),
                ),
            ]
        ),
        'linear_svc': Pipeline(
            steps=[
                ('scaler', StandardScaler()),
                (
                    'clf',
                    LinearSVC(
                        class_weight='balanced',
                        random_state=random_state,
                    ),
                ),
            ]
        ),
        'random_forest': RandomForestClassifier(
            n_estimators=500,
            class_weight='balanced_subsample',
            random_state=random_state,
            n_jobs=-1,
        ),
    }


def evaluate_predictions(y_true: pd.Series, y_pred: pd.Series) -> dict[str, float]:
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
        'recall_weighted': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }


def load_base_split_frame(repo_root: Path) -> pd.DataFrame:
    source = repo_root / 'extracted_features' / 'text' / 'lda_topics.csv'
    if not source.exists():
        raise FileNotFoundError(f'Missing LDA feature CSV: {source}')

    df = pd.read_csv(source, usecols=BASE_METADATA_COLS)
    df['path'] = df['path'].astype(str).str.replace('\\', '/', regex=False).str.strip()
    df['emotion'] = df['emotion'].astype(str).str.strip().str.lower()
    df['split'] = df['split'].astype(str).str.strip().str.lower()
    return df


def merge_family_frames(repo_root: Path, base_df: pd.DataFrame, families: list[str]) -> tuple[pd.DataFrame, list[str]]:
    merged = base_df.copy()
    feature_columns: list[str] = []

    for family in families:
        frame, info = load_family_frame(repo_root, family, prefix_features=True)
        if frame is None:
            raise FileNotFoundError(f"Family '{family}' unavailable: {info}")
        merged = merged.merge(frame, on='path', how='left')
        family_cols = [col for col in frame.columns if col != 'path']
        feature_columns.extend(family_cols)

    merged = merged.dropna(subset=feature_columns).copy()
    return merged, feature_columns


def run_combo(
    repo_root: Path,
    base_df: pd.DataFrame,
    combo: tuple[str, ...],
    random_state: int,
    min_train_rows: int,
    min_test_rows: int,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    combo_df, feature_columns = merge_family_frames(repo_root, base_df, list(combo))
    train_df = combo_df[combo_df['split'] == 'train'].copy()
    test_df = combo_df[combo_df['split'] == 'test'].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(f'Combo {combo} has empty train/test after merge and NA filtering.')
    if len(train_df) < min_train_rows or len(test_df) < min_test_rows:
        raise ValueError(
            f'Combo {combo} has too few rows after merge: train={len(train_df)} test={len(test_df)} '
            f'(minimums: train={min_train_rows}, test={min_test_rows}).'
        )

    x_train = train_df[feature_columns]
    y_train = train_df['emotion']
    x_test = test_df[feature_columns]
    y_test = test_df['emotion']

    zoo = build_model_zoo(random_state)
    rows: list[dict[str, Any]] = []
    best_payload: dict[str, Any] | None = None
    best_score = float('-inf')

    for model_name, model in zoo.items():
        model.fit(x_train, y_train)
        preds = model.predict(x_test)
        metrics = evaluate_predictions(y_test, preds)
        report = classification_report(y_test, preds, zero_division=0, output_dict=True)

        row = {
            'combo': '+'.join(combo),
            'model': model_name,
            'train_rows': int(len(train_df)),
            'test_rows': int(len(test_df)),
            'feature_count': int(len(feature_columns)),
            **metrics,
        }
        rows.append(row)

        if metrics['f1_weighted'] > best_score:
            best_score = metrics['f1_weighted']
            best_payload = {
                'combo': '+'.join(combo),
                'model': model_name,
                'metrics': metrics,
                'classification_report': report,
                'feature_count': int(len(feature_columns)),
                'train_rows': int(len(train_df)),
                'test_rows': int(len(test_df)),
            }

    if best_payload is None:
        raise RuntimeError(f'No successful model runs for combo {combo}')

    return pd.DataFrame(rows), best_payload


## 3) Load Base LDA Metadata and Inspect Split

Load the LDA feature frame metadata, remove excluded emotions, and inspect the split distribution used for all combos.

In [3]:
base_df = load_base_split_frame(REPO_ROOT)
base_df = base_df[~base_df['emotion'].isin(EXCLUDED_EMOTIONS)].copy()

print(f'Rows after emotion filter: {len(base_df)}')
print('Split distribution:')
print(base_df['split'].value_counts())
print('\nEmotion distribution:')
print(base_df['emotion'].value_counts())

base_df.head()


Rows after emotion filter: 216
Split distribution:
split
train    172
test      44
Name: count, dtype: int64

Emotion distribution:
emotion
neu    68
fru    51
xxx    44
ang    33
sad    11
exc     5
hap     4
Name: count, dtype: int64


,path,session,method,gender,emotion,n_annotators,agreement,utt_id,text,split
0,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,neu,3,3,Ses01F_script02_1_F000,fine.,test
1,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,fru,3,2,Ses01F_script02_1_F001,NaN,train
2,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,xxx,0,0,Ses01F_script02_1_F002,what?,test
3,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,neu,3,2,Ses01F_script02_1_F004,that's not your flashlight.,train
4,Session1/sentences/wav/Ses01F_script02_1/Ses01...,1,script,F,xxx,0,0,Ses01F_script02_1_F005,it's ours; it's my flashlight too.,train


## 4) Evaluate Feature-Family Combos

Run the classical-model comparison for each configured combo. Combos with too little overlap after merging are skipped.

In [4]:
all_results: list[pd.DataFrame] = []
best_payloads: list[dict[str, Any]] = []

for combo in DEFAULT_COMBOS:
    try:
        combo_results, best_payload = run_combo(
            REPO_ROOT,
            base_df,
            combo,
            random_state=RANDOM_STATE,
            min_train_rows=MIN_TRAIN_ROWS,
            min_test_rows=MIN_TEST_ROWS,
        )
    except ValueError as exc:
        print(f"[{'+'.join(combo)}] skipped: {exc}")
        continue

    all_results.append(combo_results)
    best_payloads.append(best_payload)
    best_metrics = best_payload['metrics']
    print(
        f"[{best_payload['combo']}] best={best_payload['model']} "
        f"f1_weighted={best_metrics['f1_weighted']:.4f} "
        f"accuracy={best_metrics['accuracy']:.4f}"
    )

if not all_results:
    raise RuntimeError('No valid combos were evaluated. Adjust combo selection or row thresholds.')


[lda_topics] best=random_forest f1_weighted=0.1881 accuracy=0.1591
[lda_topics_context] best=linear_svc f1_weighted=0.3375 accuracy=0.3636
[lda_topics+mfcc_normalized] best=logreg f1_weighted=0.3466 accuracy=0.3636
[lda_topics+prosody_energy+prosody_pitch] skipped: Combo ('lda_topics', 'prosody_energy', 'prosody_pitch') has too few rows after merge: train=13 test=1 (minimums: train=30, test=10).
[lda_topics+mfcc_normalized+prosody_energy+prosody_pitch] skipped: Combo ('lda_topics', 'mfcc_normalized', 'prosody_energy', 'prosody_pitch') has too few rows after merge: train=13 test=1 (minimums: train=30, test=10).
[lda_topics_context+mfcc_normalized+prosody_energy+prosody_pitch] skipped: Combo ('lda_topics_context', 'mfcc_normalized', 'prosody_energy', 'prosody_pitch') has too few rows after merge: train=13 test=1 (minimums: train=30, test=10).


## 5) Results Table and Saved Artifacts

Aggregate the combo results, sort by weighted F1, save the CSV/JSON artifacts, and display the summary table.

In [5]:
results_df = pd.concat(all_results, ignore_index=True).sort_values(
    ['f1_weighted', 'accuracy'],
    ascending=[False, False],
    ignore_index=True,
)

results_csv = OUT_DIR / 'results.csv'
best_json = OUT_DIR / 'best_models.json'

results_df.to_csv(results_csv, index=False)
best_json.write_text(json.dumps(best_payloads, indent=2), encoding='utf-8')

print(f'Saved results: {results_csv}')
print(f'Saved best-model summaries: {best_json}')

results_df


Saved results: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/feature_test/TreeBased/artifacts/text_lda_topics_eval/results.csv
Saved best-model summaries: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/feature_test/TreeBased/artifacts/text_lda_topics_eval/best_models.json


,combo,model,train_rows,test_rows,feature_count,accuracy,precision_weighted,recall_weighted,f1_weighted,f1_macro
0,lda_topics+mfcc_normalized,logreg,172,44,1182,0.363636,0.332979,0.363636,0.346585,0.213776
1,lda_topics_context,linear_svc,172,44,32,0.363636,0.329545,0.363636,0.337542,0.255230
2,lda_topics+mfcc_normalized,random_forest,172,44,1182,0.363636,0.304798,0.363636,0.318510,0.198546
3,lda_topics+mfcc_normalized,linear_svc,172,44,1182,0.295455,0.268502,0.295455,0.279395,0.177901
4,lda_topics_context,logreg,172,44,32,0.272727,0.255854,0.272727,0.256257,0.168521
5,lda_topics_context,random_forest,172,44,32,0.250000,0.222619,0.250000,0.228572,0.136606
6,lda_topics,random_forest,172,44,32,0.159091,0.242045,0.159091,0.188131,0.112245
7,lda_topics,logreg,172,44,32,0.113636,0.301136,0.113636,0.157713,0.089466
8,lda_topics,linear_svc,172,44,32,0.090909,0.220130,0.090909,0.122790,0.073413


## 6) Best Model Details

Inspect the best model selected for each valid combo.

In [6]:
pd.DataFrame([
    {
        'combo': payload['combo'],
        'model': payload['model'],
        'feature_count': payload['feature_count'],
        'train_rows': payload['train_rows'],
        'test_rows': payload['test_rows'],
        **payload['metrics'],
    }
    for payload in best_payloads
]).sort_values(['f1_weighted', 'accuracy'], ascending=[False, False], ignore_index=True)


,combo,model,feature_count,train_rows,test_rows,accuracy,precision_weighted,recall_weighted,f1_weighted,f1_macro
0,lda_topics+mfcc_normalized,logreg,1182,172,44,0.363636,0.332979,0.363636,0.346585,0.213776
1,lda_topics_context,linear_svc,32,172,44,0.363636,0.329545,0.363636,0.337542,0.255230
2,lda_topics,random_forest,32,172,44,0.159091,0.242045,0.159091,0.188131,0.112245
